# Python File Handling: Normal Text Files

A detailed, hands-on guide to persistent text files using Python's built-in `open()` and `pathlib`.

**Learning goals**
- Explain why files are useful and how a file path is resolved.
- Read, write, append, and create text files safely.
- Choose modes, use UTF-8, understand the file pointer, and handle common errors.
- Process large text files line by line.

The examples create a `file_handling_data` folder in the notebook's current working directory. They use sample data, so no personal files are needed. The reference PDF introduces text files, modes, `read()`/`readline()`/`readlines()`, `with open()`, `tell()`/`seek()`, and basic errors.

## 1. Why use files?

Variables usually live in memory while a program runs. A file preserves information after the program ends. Text files are suitable for notes, logs, configuration, reports, and simple records. A text file stores characters encoded into bytes; specifying an encoding such as UTF-8 makes the conversion predictable.

Typical workflow: **open → read or write → close**. The `with` statement closes the file automatically, even if an error happens inside the block.

In [1]:
from pathlib import Path

DATA_DIR = Path.cwd() / "file_handling_data"
DATA_DIR.mkdir(exist_ok=True)
NOTES_PATH = DATA_DIR / "notes.txt"

print("Notebook folder:", Path.cwd())
print("Data folder:", DATA_DIR.resolve())
print("Text file:", NOTES_PATH.name)

Notebook folder: c:\Users\ashir\OneDrive\Desktop\Mirai\AKTU\Python\File_handling
Data folder: C:\Users\ashir\OneDrive\Desktop\Mirai\AKTU\Python\File_handling\file_handling_data
Text file: notes.txt


## 2. File modes

| Mode | Meaning | If the file exists | If it is missing |
|---|---|---|---|
| `r` | Read text | Reads it | Raises `FileNotFoundError` |
| `w` | Write text | **Replaces** its contents | Creates it |
| `a` | Append text | Adds at the end | Creates it |
| `x` | Create only | Raises `FileExistsError` | Creates it |
| `r+` | Read and write | Opens without truncating | Raises `FileNotFoundError` |
| `t` | Text mode modifier | Default mode | Example: `rt`, `wt` |
| `b` | Binary modifier | For bytes | Example: `rb`, `wb` |

`w` can erase existing content, so use it deliberately. `a` preserves existing content. For ordinary text files, always use `encoding="utf-8"` explicitly.

## 3. Write, then read a complete file

`write()` returns the number of characters written. It does **not** add a newline for you, so include `\n` when separate lines are desired.

In [2]:
with open(NOTES_PATH, "w", encoding="utf-8") as file:
    file.write("File handling notes\n")
    file.write("Files preserve data after a program ends.\n")
    file.write("UTF-8 supports text such as café and नमस्ते.\n")

with open(NOTES_PATH, "r", encoding="utf-8") as file:
    contents = file.read()

print(contents)
print("Character count:", len(contents))

File handling notes
Files preserve data after a program ends.
UTF-8 supports text such as café and नमस्ते.

Character count: 107


## 4. Append without replacing earlier content

Opening in `a` mode writes at the end. This is useful for logs and event histories. Re-run the cell to see another line added.

In [3]:
with open(NOTES_PATH, "a", encoding="utf-8") as file:
    file.write("This line was appended.\n")

print(NOTES_PATH.read_text(encoding="utf-8"))

File handling notes
Files preserve data after a program ends.
UTF-8 supports text such as café and नमस्ते.
This line was appended.



## 5. Read using `read()`, `readline()`, and `readlines()`

- `read()` returns all remaining content as one string. Good for small files.
- `readline()` returns one line per call; a returned line usually includes `\n`.
- `readlines()` returns a list of all remaining lines; that can use substantial memory for a large file.
- `for line in file` streams lines and is usually the best choice for large files.

In [4]:
with open(NOTES_PATH, "r", encoding="utf-8") as file:
    first_line = file.readline()
    second_line = file.readline()
    print("First:", repr(first_line))
    print("Second:", repr(second_line))

with open(NOTES_PATH, "r", encoding="utf-8") as file:
    all_lines = file.readlines()
    print("Line list:", all_lines)

print("Streamed lines:")
with open(NOTES_PATH, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(line_number, line.rstrip("\n"))

First: 'File handling notes\n'
Second: 'Files preserve data after a program ends.\n'
Line list: ['File handling notes\n', 'Files preserve data after a program ends.\n', 'UTF-8 supports text such as café and नमस्ते.\n', 'This line was appended.\n']
Streamed lines:
1 File handling notes
2 Files preserve data after a program ends.
3 UTF-8 supports text such as café and नमस्ते.
4 This line was appended.


## 6. File position: `tell()` and `seek()`

A file object tracks its current position. `tell()` reports that position; `seek(0)` returns to the beginning. In text mode, positions are opaque seek positions: seeking to zero is a simple, portable reset.

In [5]:
with open(NOTES_PATH, "r", encoding="utf-8") as file:
    print("Start position:", file.tell())
    print("First 10 characters:", repr(file.read(10)))
    print("Position after reading:", file.tell())
    file.seek(0)
    print("After seek(0):", repr(file.read(10)))

Start position: 0
First 10 characters: 'File handl'
Position after reading: 10
After seek(0): 'File handl'


## 7. Safe creation and common errors

`FileNotFoundError` commonly means a read path is wrong. `PermissionError` means the process cannot access the path. Catch only errors your program can sensibly respond to; do not silently hide every exception. Use `x` when overwriting an existing file would be a mistake.

In [6]:
missing_path = DATA_DIR / "does_not_exist.txt"
try:
    with open(missing_path, "r", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError as error:
    print("Could not find:", error.filename)

exclusive_path = DATA_DIR / "created_once.txt"
try:
    with open(exclusive_path, "x", encoding="utf-8") as file:
        file.write("Created only if it did not already exist.\n")
except FileExistsError:
    print("The file already exists; x mode protected its contents.")

print("Exclusive-create example:", exclusive_path.read_text(encoding="utf-8").strip())

Could not find: c:\Users\ashir\OneDrive\Desktop\Mirai\AKTU\Python\File_handling\file_handling_data\does_not_exist.txt
Exclusive-create example: Created only if it did not already exist.


## 8. A practical example: simple student report

For more complex tabular data, use CSV rather than inventing a delimiter format. Here we write a human-readable report with a loop.

In [7]:
students = [("Rahul", 85), ("Priya", 92), ("Aman", 78)]
report_path = DATA_DIR / "marks_report.txt"

with open(report_path, "w", encoding="utf-8") as file:
    file.write("Student marks report\n")
    file.write("=====================\n")
    for name, marks in students:
        file.write(f"{name}: {marks} marks\n")

print(report_path.read_text(encoding="utf-8"))

Student marks report
Rahul: 85 marks
Priya: 92 marks
Aman: 78 marks



## 9. Summary and practice

**Remember:** choose `r`, `w`, `a`, or `x` based on intent; use `with open(..., encoding="utf-8")`; `w` replaces, `a` appends, and `write()` does not add line breaks automatically. Use iteration for large files.

1. What happens if you open an existing file with `w`? How would you preserve its old content?
2. Which method reads a single line? What does `readlines()` return?
3. Why is `with open()` safer than manually remembering `close()`?
4. Change the report example to calculate and write the average mark.
5. Make a copy of `notes.txt` by reading it and writing the contents to a second UTF-8 text file.